# 📝 Week 4 Homework: Data Wrangling - From Business Question to Analysis

<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-7025/blob/main/assignments/homework/week-04-homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

## 📂 Instructions

This homework is based on the Lab we worked through in Thursday's class.  So, if you completed that Lab, you can use that notebook for the homework.

Complete the tasks below in this Jupyter notebook. Most tasks require you to write Python code and use the output to answer **a separate online quiz**.

At the end, you’ll also upload this completed `.ipynb` notebook

---

In this homework, we’ll use **three datasets** from the Complete Journey retail grocery data:

1. **transactions** – product purchases by households (receipt-level detail)  
2. **demographics** – household-level demographic data  
3. **products** – metadata about products purchased  

This homework reinforces this week’s readings:

- **[Reading 10: Manipulating Data](https://bradleyboehmke.github.io/uc-bana-7025/10-manipulating-data.html)**
- **[Reading 11: Summarizing Data](https://bradleyboehmke.github.io/uc-bana-7025/11_aggregating_data.html)**
- **[Reading 12: Joining Data](https://bradleyboehmke.github.io/uc-bana-7025/12-joining-data.html)**

We will:
- Start with simple data exploration
- Progress to manipulating and summarizing data
- End with joining datasets to answer more complex questions
- Practice breaking business questions into **analytical steps**

You are encouraged to work in small groups of **2–4 students** but you must submit your own notebook.


In [1]:
# If you don't have completejourney_py installed, run: pip install completejourney-py
from completejourney_py import get_data
import pandas as pd

# Load datasets
cj_data = get_data()
transactions = cj_data['transactions']
products = cj_data['products']
demographics = cj_data['demographics']

# Quick preview
transactions.head()


ModuleNotFoundError: No module named 'completejourney_py'

In [2]:
!pip install completejourney-py
from completejourney_py import get_data
import pandas as pd

cj_data = get_data()
transactions = cj_data['transactions']
products = cj_data['products']
demographics = cj_data['demographics']

transactions.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.6/31.6 MB 36.2 MB/s eta 0:00:00


,household_id,store_id,basket_id,product_id,quantity,sales_value,retail_disc,coupon_disc,coupon_match_disc,week,transaction_timestamp
0,900,330,31198570044,1095275,1,0.50,0.00,0.0,0.0,1,2017-01-01 11:53:26
1,900,330,31198570047,9878513,1,0.99,0.10,0.0,0.0,1,2017-01-01 12:10:28
2,1228,406,31198655051,1041453,1,1.43,0.15,0.0,0.0,1,2017-01-01 12:26:30
3,906,319,31198705046,1020156,1,1.50,0.29,0.0,0.0,1,2017-01-01 12:30:27
4,906,319,31198705046,1053875,2,2.78,0.80,0.0,0.0,1,2017-01-01 12:30:27


## Setup

## Part 1 – Basic Exploration


**Q0:** How many transactions are in our dataset, what is the date range, how many households have demographic data, how many products exist, and what are the min/max/mean sales values?  

**Step-by-step instructions:**
1. Use `.shape[0]` on `transactions` to count rows.  
2. Use `.min()` and `.max()` on `transaction_timestamp` to find the date range.  
3. Use `.shape[0]` on `demographics` and `products` to get counts.  
4. Use `.min()`, `.max()`, `.mean()` on `sales_value` for basic stats.


In [23]:
# Starter code with blanks to fill
# total number of transactions
num_transactions = transactions.shape[0]
print("Number of transactions:", num_transactions)

#

Number of transactions: 1469307


In [24]:
# date range of transactions
min_date = transactions['transaction_timestamp'].min()
max_date = transactions['transaction_timestamp'].max()
print("Date range:", min_date, "to", max_date)

Date range: 2017-01-01 11:53:26 to 2018-01-01 04:01:20


In [25]:
# number of unique households and products
num_households = demographics.shape[0]
num_products = products.shape[0]
print("Number of households:", num_households)
print("Number of products:", num_products)


Number of households: 801
Number of products: 92331


In [26]:
# summary statistics for sales_value
min_sales = transactions['sales_value'].min()
max_sales = transactions['sales_value'].max()
mean_sales = transactions['sales_value'].mean()
print("Min sales:", min_sales)
print("Max sales:", max_sales)
print("Mean sales:", mean_sales)


Min sales: 0.0
Max sales: 840.0
Mean sales: 3.12803218115751



**Q1:** Which day had the highest total sales?  

**Step-by-step instructions:**
1. Create a new column `date` by extracting only the date from `transaction_timestamp` (`.dt.date`).  
2. Group by `date` and sum `sales_value`.  
3. Sort results in descending order.  
4. Select the top row.


In [7]:
# Your code here
transactions['date'] = transactions['transaction_timestamp'].dt.date

top_sales_day = (
    transactions
    .groupby('date')['sales_value']
    .sum()
    .sort_values(ascending=False)
    .head(1)
)
print(top_sales_day)

date
2017-12-23    24994.47
Name: sales_value, dtype: float64



**Q2:** What are the top 5 departments by total sales?  

**Step-by-step instructions:**
1. Join `transactions` to `products` on `product_id` using an inner join.  
2. Group by `department` and sum `sales_value`.  
3. Sort results in descending order.  
4. Display the top 5.


In [8]:
# Your code here
top_departments = (
    transactions
    .merge(products, on='product_id', how='inner')
    .groupby('department')['sales_value']
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
print(top_departments)

department
GROCERY    2316393.89
DRUG GM     596827.45
FUEL        329594.45
PRODUCE     322858.82
MEAT        308575.33
Name: sales_value, dtype: float64


## Part 2 – Manipulating Data


**Q3:** What is the average unit price for each department?  

**Step-by-step instructions:**
1. Create a `unit_price` column: `sales_value / quantity`.  
2. Join `transactions` to `products` to bring in `department`.  
3. Group by `department` and calculate the mean of `unit_price`.


In [9]:
# Your code here
transactions['unit_price'] = transactions['sales_value'] / transactions['quantity']

avg_unit_price = (
    transactions
    .merge(products, on='product_id', how='inner')
    .groupby('department')['unit_price']
    .mean()
)
print(avg_unit_price)

department
AUTOMOTIVE          7.216111
CHEF SHOPPE         2.522274
CNTRL/STORE SUP     3.150000
COSMETICS           4.138923
COUPON              1.296070
DELI                     inf
DRUG GM                  inf
ELECT &PLUMBING     1.000000
FLORAL              7.732635
FROZEN GROCERY      2.794859
FUEL                1.516282
GARDEN CENTER            inf
GM MERCH EXP        1.572222
GROCERY                  inf
MEAT                     inf
MEAT-PCKGD               inf
MISCELLANEOUS       4.363352
NUTRITION           2.483601
PASTRY              2.895513
PHOTO & VIDEO       2.835882
POSTAL CENTER       1.156000
PROD-WHS SALES      0.840000
PRODUCE                  inf
RESTAURANT          3.704349
SALAD BAR           2.939270
SEAFOOD             6.721697
SEAFOOD-PCKGD       4.509113
SPIRITS             9.730135
TOYS                1.290000
TRAVEL & LEISURE    2.853138
Name: unit_price, dtype: float64



**Q4:** Do we have missing values in `unit_price`?  

**Step-by-step instructions:**
1. Use `.isna().sum()` on `unit_price` to count missing values.  
2. Consider filtering rows where `quantity == 0` to see if that’s the cause.


In [10]:
# Your code here
# Count NaN / null values
missing_count = transactions['unit_price'].isna().sum()
print("Missing unit prices:", missing_count)

# Check zero-quantity rows causing division by zero or NaN
zero_qty = (transactions['quantity'] == 0).sum()
print("Transactions with quantity == 0:", zero_qty)

Missing unit prices: 8820
Transactions with quantity == 0: 8869


## Part 3 – Aggregations


**Q5:** Which income level spends the most on average?

*Hint:* Join transactions to demographics, group by income, calculate mean sales per income level.


In [11]:
# Your code here
income_spend = (
    transactions
    .merge(demographics, on='household_id', how='inner')
    .groupby('income')['sales_value']
    .mean()
    .sort_values(ascending=False)
)
print(income_spend)

income
175-199K     3.754513
250K+        3.724832
200-249K     3.703222
150-174K     3.541206
100-124K     3.481148
125-149K     3.457844
75-99K       3.327325
50-74K       3.149264
35-49K       2.978827
Under 15K    2.977735
25-34K       2.957803
15-24K       2.921428
Name: sales_value, dtype: float64



**Q6:** Do households with kids spend more (on average) than households without kids?  

*Hint:* Use `kid_count` to group households by creating a new column (e.g., `has_kids`) that identifies whether a household has kids (`kid_count > 0`) or not (`kid_count == 0`). Note that the tricky part of this step is that `kid_count` is not a numeric variable 🤔. Compute the average spend for those with kids and those without.


In [16]:
# Your code here
# Inspect unique values in kids_count
print("Original kids_count value counts:")
print(demographics['kids_count'].value_counts(dropna=False))

# Convert 'kids_count' to a numerical representation
# Replace '3+' and '5+' with their numerical equivalents
demographics['kids_count_numeric'] = demographics['kids_count'].replace({'3+': '3', '5+': '5'})
# Coerce to numeric, turning non-convertible values (like None) into NaN
demographics['kids_count_numeric'] = pd.to_numeric(demographics['kids_count_numeric'], errors='coerce')

print("\nNumeric kids_count value counts:")
print(demographics['kids_count_numeric'].value_counts(dropna=False))

# Create indicator: True if kids are present (kids_count_numeric > 0)
demographics['has_kids'] = demographics['kids_count_numeric'] > 0

# Calculate average sales_value for households with kids vs without
kids_spend = (
    transactions
    .merge(demographics, on='household_id', how='inner')
    .groupby('has_kids')['sales_value']
    .mean()
)

print("\nAverage sales value by 'has_kids':")
print(kids_spend)

Original kids_count value counts:
kids_count
0     513
1     159
3+     69
2      60
Name: count, dtype: int64

Numeric kids_count value counts:
kids_count_numeric
0    513
1    159
3     69
2     60
Name: count, dtype: int64

Average sales value by 'has_kids':
has_kids
False    3.177327
True     3.150616
Name: sales_value, dtype: float64



**Q7:** What are the top 5 departments by total quantity of items sold?  

*Hint:* Join to products, group by department, sum quantity, and sort.


In [17]:
# Your code here
top_dept_qty = (
    transactions
    .merge(products, on='product_id', how='inner')
    .groupby('department')['quantity']
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
print(top_dept_qty)

department
FUEL             129662940
MISCELLANEOUS     21361882
GROCERY            1242944
DRUG GM             198635
PRODUCE             185444
Name: quantity, dtype: int64


## Part 4 – Joins for Deeper Insights


**Q8:** Which product is purchased most frequently?  

*Hint:* Group by `product_id`, sum quantity, then join to products for description.


In [18]:
# Your code here
most_frequent = (
    transactions
    .groupby('product_id')['quantity']
    .sum()
    .reset_index()
    .merge(products, on='product_id', how='inner')
    .sort_values(by='quantity', ascending=False)
    .head(1)
)
print(most_frequent[['product_id', 'product_category', 'product_type', 'quantity']])

       product_id   product_category           product_type   quantity
42352     6534178  COUPON/MISC ITEMS  GASOLINE-REG UNLEADED  126868510



**Q9:** Identify all products with “pizza” in `product_type` and find the one with the greatest total sales.  

*Hint:* Filter products where product_type contains "pizza" with `.str.contains("pizza", case=False, na=False)`, join to transactions, sum sales by product.


In [19]:
# Your code here
pizza_products = products[products['product_type'].str.contains('pizza', case=False, na=False)]

top_pizza = (
    transactions
    .merge(pizza_products, on='product_id', how='inner')
    .groupby(['product_id', 'product_type'])['sales_value']
    .sum()
    .sort_values(ascending=False)
    .head(1)
)
print(top_pizza)

product_id  product_type     
944139      PIZZA/TRADITIONAL    1344.5
Name: sales_value, dtype: float64



**Q10:** Which product category brings in the most revenue for the highest-income households with kids?

*Hint:* Filter demographics for the highest income level & `kid_count > 0`, join to transactions and products, group by category and compute the sum of sales value.


In [27]:
# Your code here
# Check exact format of highest income bracket (e.g., '250K+' or '$250k+')
print(demographics['income'].unique())

highest_income_val = demographics['income'].dropna().max()  # verify against quiz options
high_inc_kids = demographics[(demographics['income'] == highest_income_val) & (demographics['has_kids'] == True)]

top_category = (
    transactions
    .merge(high_inc_kids, on='household_id', how='inner')
    .merge(products, on='product_id', how='inner')
    .groupby('product_category')['sales_value']
    .sum()
    .sort_values(ascending=False)
    .head(1)
)
print(top_category)

['35-49K' '50-74K' '25-34K' '15-24K' 'Under 15K' '75-99K' '100-124K'
 '125-149K' '150-174K' '250K+' '175-199K' '200-249K']
product_category
SOFT DRINKS    5614.83
Name: sales_value, dtype: float64



**Q11:** Which manufacturer has the highest total sales, and which department do they primarily sell in?  

*Hint:* Join transactions to products, group by manufacturer, sum sales, find top. Then, filter products for that top manufacturer and check which department(s) they are associated with.


In [21]:
# Your code here
top_mfr_sales = (
    transactions
    .merge(products, on='product_id', how='inner')
    .groupby('manufacturer_id')['sales_value']
    .sum()
    .sort_values(ascending=False)
    .head(1)
)
top_mfr_id = top_mfr_sales.index[0]

top_dept_for_mfr = (
    products[products['manufacturer_id'] == top_mfr_id]['department']
    .value_counts()
    .head(1)
)
print(f"Top Manufacturer ID: {top_mfr_id}")
print("Primary Department:")
print(top_dept_for_mfr)

Top Manufacturer ID: 69
Primary Department:
department
GROCERY    8704
Name: count, dtype: int64



**Q12:** For each income level, what is the most frequently purchased product category?  

*Hint:* Join demographics → transactions → products, group by income & category, count quantity, get top per income.


In [22]:
# Your code here
cat_by_income = (
    transactions
    .merge(demographics, on='household_id', how='inner')
    .merge(products, on='product_id', how='inner')
    .groupby(['income', 'product_category'])['quantity']
    .sum()
    .reset_index()
)

# Extract category with max quantity per income level
idx = cat_by_income.groupby('income')['quantity'].idxmax()
top_cat_per_income = cat_by_income.loc[idx].sort_values('income')
print(top_cat_per_income)

         income   product_category  quantity
69     100-124K  COUPON/MISC ITEMS   6489551
348    125-149K  COUPON/MISC ITEMS   7753726
628      15-24K  COUPON/MISC ITEMS   5157242
908    150-174K  COUPON/MISC ITEMS   6352923
1176   175-199K  COUPON/MISC ITEMS   1843471
1424   200-249K  COUPON/MISC ITEMS    105087
1657     25-34K  COUPON/MISC ITEMS   6570244
1937      250K+  COUPON/MISC ITEMS   1096221
2211     35-49K  COUPON/MISC ITEMS  16342739
2505     50-74K  COUPON/MISC ITEMS  25842845
2798     75-99K  COUPON/MISC ITEMS  13267153
3081  Under 15K  COUPON/MISC ITEMS   6076980


## Homework Deliverable


- Implement the code to answer the above questions.  
- Once you have all your answers, go to the homework quiz on Canvas and submit your answers.  
- Save your notebook — you'll upload it on Canvas as part of the homework.
